# Pipeline Helper
Utilities for working with `analysis.json` files: reading, inspecting scenes/subsegments, joining LLM evaluations, and generating VLM prompts.

In [ ]:
import json
import subprocess
from pathlib import Path
from typing import Any

## Config

In [2]:
ANALYSIS_PATH = Path("../local/videos/DJI_20250514105245_0135_D.analysis.json")

## I/O

In [3]:
def read_analysis(path: str | Path) -> dict:
    """Load an analysis.json file and return the parsed dict."""
    with open(path) as f:
        return json.load(f)

## Scene attribute extraction

In [4]:
DEFAULT_SCENE_ATTRS = ["start_time", "end_time"]


def _get_nested(obj: dict, key: str) -> Any:
    """Resolve a dotted key path like 'channel_metrics.pan.rms' from a dict."""
    for part in key.split("."):
        if not isinstance(obj, dict) or part not in obj:
            return None
        obj = obj[part]
    return obj


def extract_scene_attributes(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
) -> list[dict]:
    """
    Extract specified attributes from each scene.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    attributes:
        Scene-level keys to include.  Dotted paths are supported,
        e.g. ``["start_time", "end_time", "quality_score"]``.

    Returns
    -------
    List of dicts, one per scene, always including ``scene_id``.
    """
    rows = []
    for scene in analysis.get("scenes", []):
        row = {"scene_id": scene["scene_id"]}
        for attr in attributes:
            row[attr] = _get_nested(scene, attr)
        rows.append(row)
    return rows


def print_scenes_json(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
    indent: int = 2,
) -> None:
    """Print per-scene attributes as a JSON array."""
    rows = extract_scene_attributes(analysis, attributes)
    print(json.dumps(rows, indent=indent))


def print_scenes_table(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
    float_fmt: str = ".3f",
) -> None:
    """Print per-scene attributes as a plain-text table."""
    rows = extract_scene_attributes(analysis, attributes)
    if not rows:
        print("(no scenes)")
        return

    columns = list(rows[0].keys())

    # Format all values as strings
    str_rows = []
    for row in rows:
        str_row = {}
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                str_row[col] = format(val, float_fmt)
            else:
                str_row[col] = str(val) if val is not None else ""
        str_rows.append(str_row)

    # Column widths
    widths = {col: max(len(col), *(len(r[col]) for r in str_rows)) for col in columns}

    sep = "+" + "+".join("-" * (widths[c] + 2) for c in columns) + "+"
    header = "|" + "|".join(f" {c.ljust(widths[c])} " for c in columns) + "|"

    print(sep)
    print(header)
    print(sep)
    for row in str_rows:
        line = "|" + "|".join(f" {row[c].ljust(widths[c])} " for c in columns) + "|"
        print(line)
    print(sep)

### Demo – scenes

In [5]:
analysis = read_analysis(ANALYSIS_PATH)

print_scenes_json(analysis)
print()
print_scenes_table(analysis)

[
  {
    "scene_id": 0,
    "start_time": 0.0,
    "end_time": 12.75
  },
  {
    "scene_id": 1,
    "start_time": 12.75,
    "end_time": 61.5
  },
  {
    "scene_id": 4,
    "start_time": 96.5,
    "end_time": 110.25
  },
  {
    "scene_id": 3,
    "start_time": 74.0,
    "end_time": 96.5
  },
  {
    "scene_id": 2,
    "start_time": 61.5,
    "end_time": 74.0
  },
  {
    "scene_id": 5,
    "start_time": 110.25,
    "end_time": 120.0
  }
]

+----------+------------+----------+
| scene_id | start_time | end_time |
+----------+------------+----------+
| 0        | 0.000      | 12.750   |
| 1        | 12.750     | 61.500   |
| 4        | 96.500     | 110.250  |
| 3        | 74.000     | 96.500   |
| 2        | 61.500     | 74.000   |
| 5        | 110.250    | 120.000  |
+----------+------------+----------+


## Subsegment attribute extraction

In [6]:
DEFAULT_SUBSEG_ATTRS = ["start_time", "end_time"]


def extract_subsegment_attributes(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
) -> list[dict]:
    """
    Extract specified attributes from every subsegment across all scenes.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    attributes:
        Subsegment-level keys to include.  Dotted paths are supported,
        e.g. ``["start_time", "end_time", "channel_metrics.pan.rms"]``.

    Returns
    -------
    List of dicts, one per subsegment, always including ``scene_id``
    and ``subsegment_id``.
    """
    rows = []
    for scene in analysis.get("scenes", []):
        for sub in scene.get("subsegments", []):
            row = {
                "scene_id": scene["scene_id"],
                "subsegment_id": sub["subsegment_id"],
            }
            for attr in attributes:
                row[attr] = _get_nested(sub, attr)
            rows.append(row)
    return rows


def print_subsegments_json(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
    indent: int = 2,
) -> None:
    """Print per-subsegment attributes as a JSON array."""
    rows = extract_subsegment_attributes(analysis, attributes)
    print(json.dumps(rows, indent=indent))


def print_subsegments_table(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
    float_fmt: str = ".3f",
) -> None:
    """Print per-subsegment attributes as a plain-text table."""
    rows = extract_subsegment_attributes(analysis, attributes)
    if not rows:
        print("(no subsegments)")
        return

    columns = list(rows[0].keys())

    str_rows = []
    for row in rows:
        str_row = {}
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                str_row[col] = format(val, float_fmt)
            else:
                str_row[col] = str(val) if val is not None else ""
        str_rows.append(str_row)

    widths = {col: max(len(col), *(len(r[col]) for r in str_rows)) for col in columns}

    sep = "+" + "+".join("-" * (widths[c] + 2) for c in columns) + "+"
    header = "|" + "|".join(f" {c.ljust(widths[c])} " for c in columns) + "|"

    print(sep)
    print(header)
    print(sep)
    for row in str_rows:
        line = "|" + "|".join(f" {row[c].ljust(widths[c])} " for c in columns) + "|"
        print(line)
    print(sep)

### Demo – subsegments

In [7]:
print_subsegments_json(analysis)
print()
print_subsegments_table(analysis)

[
  {
    "scene_id": 0,
    "subsegment_id": 0,
    "start_time": 0.5,
    "end_time": 12.5
  },
  {
    "scene_id": 1,
    "subsegment_id": 0,
    "start_time": 12.75,
    "end_time": 15.0
  },
  {
    "scene_id": 1,
    "subsegment_id": 1,
    "start_time": 15.0,
    "end_time": 40.0
  },
  {
    "scene_id": 1,
    "subsegment_id": 2,
    "start_time": 40.0,
    "end_time": 47.5
  },
  {
    "scene_id": 1,
    "subsegment_id": 3,
    "start_time": 47.5,
    "end_time": 55.0
  },
  {
    "scene_id": 1,
    "subsegment_id": 4,
    "start_time": 55.0,
    "end_time": 57.5
  },
  {
    "scene_id": 1,
    "subsegment_id": 5,
    "start_time": 57.5,
    "end_time": 61.25
  },
  {
    "scene_id": 4,
    "subsegment_id": 0,
    "start_time": 96.5,
    "end_time": 97.5
  },
  {
    "scene_id": 4,
    "subsegment_id": 1,
    "start_time": 97.5,
    "end_time": 102.5
  },
  {
    "scene_id": 4,
    "subsegment_id": 2,
    "start_time": 102.5,
    "end_time": 106.25
  },
  {
    "scene_id": 4,


## Join LLM evaluation

In [8]:
def join_llm_evaluation(
    analysis: dict,
    llm_json: str | Path | list[dict],
    join_key: str = "scene_id",
    field_prefix: str = "llm_",
) -> dict:
    """
    Merge LLM evaluation results into the analysis dict (in-place copy).

    The LLM JSON must be a list of objects each containing ``join_key``
    (default ``scene_id``).  All other fields from each LLM object are
    injected into the matching scene under the given ``field_prefix``.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.  Not mutated; a shallow-merged copy
        is returned.
    llm_json:
        Path to a JSON file **or** an already-parsed list of dicts.
    join_key:
        The field used to match LLM records to scenes (default ``scene_id``).
    field_prefix:
        Prefix added to every field imported from the LLM evaluation
        to avoid name collisions (default ``llm_``).

    Returns
    -------
    A new analysis dict with LLM fields merged into each scene.
    """
    if not isinstance(llm_json, list):
        with open(llm_json) as f:
            llm_records = json.load(f)
    else:
        llm_records = llm_json

    llm_by_key = {rec[join_key]: rec for rec in llm_records if join_key in rec}

    merged_scenes = []
    for scene in analysis.get("scenes", []):
        scene_copy = dict(scene)
        key_val = scene.get(join_key)
        if key_val in llm_by_key:
            for k, v in llm_by_key[key_val].items():
                if k != join_key:
                    scene_copy[f"{field_prefix}{k}"] = v
        merged_scenes.append(scene_copy)

    return {**analysis, "scenes": merged_scenes}

### Demo – join LLM evaluation

In [9]:
# Example: provide a list of dicts directly (or pass a path to a JSON file)
example_llm_eval = [
    {"scene_id": 0, "description": "Slow pan over landscape", "keep": True, "rating": 4},
    {"scene_id": 1, "description": "Fast zoom in",            "keep": False, "rating": 2},
]

merged = join_llm_evaluation(analysis, example_llm_eval)

# Verify the new fields appear in the first two scenes
for scene in merged["scenes"][:3]:
    llm_fields = {k: v for k, v in scene.items() if k.startswith("llm_")}
    print(f"scene {scene['scene_id']}: {llm_fields}")

scene 0: {'llm_description': 'Slow pan over landscape', 'llm_keep': True, 'llm_rating': 4}
scene 1: {'llm_description': 'Fast zoom in', 'llm_keep': False, 'llm_rating': 2}
scene 4: {}


## Extract analysis subset (strip per-frame data)

In [10]:
# Top-level keys that contain per-frame arrays – excluded from the subset
_PER_FRAME_KEYS = {"frame_metrics", "flow_stats", "flow_decomposition"}


def extract_analysis_subset(
    analysis: dict,
    exclude_keys: set[str] | None = None,
    include_subsegments: bool = True,
) -> dict:
    """
    Return a lightweight copy of the analysis dict without per-frame arrays.

    Keeps all top-level metadata (video_path, duration, num_scenes, boundaries…)
    and the full scene list including subsegments.  Per-frame arrays
    (frame_metrics, flow_stats, flow_decomposition) are dropped by default.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    exclude_keys:
        Override the set of top-level keys to drop.  Defaults to
        ``{"frame_metrics", "flow_stats", "flow_decomposition"}``.
    include_subsegments:
        When ``False``, the ``subsegments`` list is removed from each scene
        (useful for an even more compact summary).

    Returns
    -------
    A new dict with the requested keys removed.
    """
    excluded = exclude_keys if exclude_keys is not None else _PER_FRAME_KEYS

    subset = {k: v for k, v in analysis.items() if k not in excluded}

    if not include_subsegments and "scenes" in subset:
        trimmed_scenes = []
        for scene in subset["scenes"]:
            scene_copy = {k: v for k, v in scene.items() if k != "subsegments"}
            trimmed_scenes.append(scene_copy)
        subset["scenes"] = trimmed_scenes

    return subset

### Demo – subset

In [11]:
subset = extract_analysis_subset(analysis)
print("Top-level keys kept:", list(subset.keys()))
print(f"Scenes: {len(subset['scenes'])}, first scene keys: {list(subset['scenes'][0].keys())}")

Top-level keys kept: ['video_path', 'duration', 'num_scenes', 'scenes', 'boundaries', 'flow_boundaries', 'emb_boundaries']
Scenes: 6, first scene keys: ['scene_id', 'start_time', 'end_time', 'duration', 'avg_sharpness', 'avg_brightness', 'avg_brisque', 'avg_flow_magnitude', 'avg_flow_coherence', 'quality_score', 'keyframe_timestamps', 'emb_start', 'emb_end', 'subsegments']


## VLM prompt generation

In [12]:
def _format_time(seconds: float) -> str:
    """Format seconds as MM:SS.mmm for human-readable timestamps."""
    minutes = int(seconds) // 60
    secs = seconds - minutes * 60
    return f"{minutes:02d}:{secs:06.3f}"


def generate_vlm_prompt(
    analysis: dict,
    template: str | None = None,
    time_format: str = "seconds",
) -> str:
    """
    Build a VLM prompt that includes the pre-computed scene segmentation.

    The scene boundaries are embedded directly so the VLM does **not** need
    to perform its own segmentation – it should evaluate the provided scenes
    as-is.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    template:
        Optional prompt template string.  Use ``{scenes}`` as the placeholder
        for the scene block and ``{num_scenes}`` for the scene count.
        If ``None``, a minimal placeholder template is used so you can see
        the scene block output and slot in the real template later.
    time_format:
        ``"seconds"`` (default) or ``"timecode"`` (MM:SS.mmm).

    Returns
    -------
    The complete prompt string.
    """
    scenes = analysis.get("scenes", [])

    lines = []
    for scene in scenes:
        sid = scene["scene_id"]
        start = scene["start_time"]
        end = scene["end_time"]

        if time_format == "timecode":
            start_str = _format_time(start)
            end_str = _format_time(end)
        else:
            start_str = f"{start:.3f}s"
            end_str = f"{end:.3f}s"

        lines.append(f"  Scene {sid}: {start_str} – {end_str}")

    scene_block = "\n".join(lines)

    if template is None:
        template = (
            "[PROMPT TEMPLATE PLACEHOLDER]\n"
            "\n"
            "The video has been pre-segmented into {num_scenes} scenes.\n"
            "Do NOT perform your own segmentation. Evaluate each scene as provided:\n"
            "\n"
            "{scenes}"
        )

    return template.format(scenes=scene_block, num_scenes=len(scenes))

### Demo – VLM prompt

In [13]:
prompt = generate_vlm_prompt(analysis, time_format="timecode")
print(prompt)

[PROMPT TEMPLATE PLACEHOLDER]

The video has been pre-segmented into 6 scenes.
Do NOT perform your own segmentation. Evaluate each scene as provided:

  Scene 0: 00:00.000 – 00:12.750
  Scene 1: 00:12.750 – 01:01.500
  Scene 4: 01:36.500 – 01:50.250
  Scene 3: 01:14.000 – 01:36.500
  Scene 2: 01:01.500 – 01:14.000
  Scene 5: 01:50.250 – 02:00.000


In [ ]:
# With a custom template (fill in the real one when ready)
custom_template = """\
**Role:** You are a Professional Video Editor and Colorist. Your task is to perform a granular "First Pass" analysis of video footage to prepare it for cutting and color matching.

**Segmentation Rules:**
* **Duration:** Scenes should not exceed 30s unless a continuous, high-value action is occurring.
* **Precision:** Create a new scene entry for every change in camera movement, significant lighting shift, or subject transition.
* **Assessment:** Be rigorous. Evaluate scenes based on professional cinematography standards (stability, framing, and narrative intent).

**Instructions for Color Analysis:**
* **Dominant Colors:** Identify the primary colors that occupy the most screen real estate.
* **Lighting Type:** Define the light source to help the editor match footage shot at different times or locations.

**Output Format:** Return **strictly valid JSON** in the following structure. Do not include introductory or concluding prose.

```json Scene object
[
  {
    "scene_id": "integer",
    "time_range": {
      "start": "HH:MM:SS.mmm",
      "end": "HH:MM:SS.mmm"
    },
    "description": "Narrative summary: [Subject] [Action] in [Environment]. (e.g., 'Medium shot of a hiker reaching the summit during golden hour.')",
    "technical_specs": {
      "framing": "Select from: Extreme Wide Shot, Wide Shot, Medium Shot, Medium Close-Up, Close-Up, Extreme Close-Up, Over-the-Shoulder, POV Shot",
      "movement": "Select from: Pan Left/Right, Tilt Up/Down, Dolly In/Out, Truck Left/Right, Pedestal Up/Down, Tracking Shot, Zoom In/Out, Dolly Zoom, Fly-By, Arc Shot",
      "angle": "Select from: Eye Level, High Angle, Low Angle, Dutch Angle, Bird’s Eye View, Worm’s Eye View",
      "execution_rating": "one of [excellent, good, medium, bad]",
      "reasoning": "Detailed reasoning explaining chosen framing, movement, and angle."
    },
    "color_profile": {
      "dominant_colors": ["hex_code or color name"],
      "lighting_type": "e.g., Natural/Golden Hour, Harsh Midday, Low-light/Interior, Neon/Stylized",
      "temperature": "one of [warm, cool, neutral]"
    },
    "highlights": [
      {
        "description": "Specific micro-moment of interest",
        "keywords": ["max 5 tags"],
        "start": "HH:MM:SS.mmm",
         "end": "HH:MM:SS.mmm",
         "reasoning": "Reason why this moment was chosen as highlight"
      }
    ],
    "quality_score": {
      "rating": "one of [excellent, good, medium, bad]",
      "reasoning": "Brief evaluation of aesthetic and cinematographic value."
    },
    "scene_tags": ["list of max 7 keywords to identify the scene"]
  }
]
```

Example JSON:
[
  {
    "scene_id": 1,
    "time_range": {
      "start": "00:00:00.000",
      "end": "00:00:05.500"
    },
    "description": "Aerial bird's-eye view of a turquoise alpine lake surrounded by jagged limestone peaks. A small red boat is centered, creating a focal point against the water.",
    "technical_specs": {
      "framing": "Wide Shot",
      "movement": "Dolly In",
      "angle": "Bird’s Eye View",
      "execution_rating": "excellent",
      "technical_notes": "Stable flight path with no micro-jitters; perfect centering of the boat subject throughout the move."
    },
    "color_profile": {
      "dominant_colors": ["#40E0D0", "#A9A9A9", "#FF0000"],
      "lighting_type": "Harsh Midday",
      "contrast_level": "high",
      "temperature": "neutral",
    },
    "highlights": [
      {
        "description": "Sunlight reflects sharply off the boat's hull",
        "keywords": ["reflection", "specular highlight", "focal point"],
        "start": "00:00:02.350",
        "start": "00:00:07.850"
      }
    ],
    "quality_score": {
      "rating": "excellent",
      "reasoning": "High dynamic range captured well; smooth gimbal movement provides professional cinematic feel with no visible compression artifacts."
    },
    "scene_tags": ["alpine", "lake", "drone", "nature", "turquoise", "serene", "aerial"]
  }
]


The video contains {num_scenes} pre-segmented scenes:
{scenes}

Respond with a JSON array with one object per scene.
"""

print(generate_vlm_prompt(analysis, template=custom_template, time_format="seconds"))

You are evaluating drone footage. For each scene listed below, provide:
- A short description of camera movement and subject
- A quality rating from 1-5
- Whether the scene should be kept (yes/no) and why

The video contains 6 pre-segmented scenes:
  Scene 0: 0.000s – 12.750s
  Scene 1: 12.750s – 61.500s
  Scene 4: 96.500s – 110.250s
  Scene 3: 74.000s – 96.500s
  Scene 2: 61.500s – 74.000s
  Scene 5: 110.250s – 120.000s

Respond with a JSON array with one object per scene.



## Scene extraction with ffmpeg

In [ ]:
def extract_scenes_to_clips(
    analysis: dict,
    output_dir: str | Path,
    video_path: str | Path | None = None,
    scene_ids: list[int] | None = None,
    codec: str = "copy",
    ext: str = "mp4",
) -> list[Path]:
    """
    Cut a video into per-scene clips using ffmpeg.

    Output structure::

        output_dir/
        └── <video_stem>/
            ├── scene_00.mp4
            ├── scene_01.mp4
            └── ...

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    output_dir:
        Root folder under which a sub-folder named after the video is created.
    video_path:
        Path to the source video.  Defaults to ``analysis["video_path"]``.
    scene_ids:
        Optional list of scene IDs to extract.  Extracts all scenes when ``None``.
    codec:
        ffmpeg ``-c`` value.  ``"copy"`` (default) is fast and lossless;
        pass ``"libx264"`` for re-encoded, frame-accurate cuts.
    ext:
        Output file extension (default ``"mp4"``).

    Returns
    -------
    List of Paths to the created clip files, sorted by scene_id.
    """
    src = Path(video_path or analysis["video_path"])
    if not src.exists():
        raise FileNotFoundError(f"Video not found: {src}")

    out_root = Path(output_dir) / src.stem
    out_root.mkdir(parents=True, exist_ok=True)

    scenes = analysis.get("scenes", [])
    if scene_ids is not None:
        scenes = [s for s in scenes if s["scene_id"] in scene_ids]
    scenes = sorted(scenes, key=lambda s: s["scene_id"])

    created = []
    for scene in scenes:
        sid = scene["scene_id"]
        start = scene["start_time"]
        end = scene["end_time"]
        duration = end - start

        out_file = out_root / f"scene_{sid:02d}.{ext}"

        cmd = [
            "ffmpeg", "-y",
            "-ss", str(start),
            "-i", str(src),
            "-t", str(duration),
            "-c", codec,
            "-avoid_negative_ts", "make_zero",
            str(out_file),
        ]

        print(f"  Extracting scene {sid}  [{start:.3f}s – {end:.3f}s]  → {out_file}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(
                f"ffmpeg failed for scene {sid}:\n{result.stderr}"
            )
        created.append(out_file)

    print(f"\nDone. {len(created)} clip(s) written to: {out_root}")
    return created

### Demo – extract all scenes

In [ ]:
OUTPUT_DIR = Path("../local/clips")

clips = extract_scenes_to_clips(analysis, output_dir=OUTPUT_DIR)

# To extract only specific scenes:
# clips = extract_scenes_to_clips(analysis, output_dir=OUTPUT_DIR, scene_ids=[0, 2])

# To force re-encode for frame-accurate cuts (slower):
# clips = extract_scenes_to_clips(analysis, output_dir=OUTPUT_DIR, codec="libx264")